# Reading Seoul Through Its Movement — Data Pipeline

This notebook is a **thin orchestrator**. All logic lives in `src/`; every cell below either calls a function from that package or shows a result. Reviewers should be able to read this notebook top-to-bottom in ~5 minutes and understand exactly what the pipeline does.

**Pipeline stages**

| Stage | What happens | Module |
|---|---|---|
| 0 | Ingest raw ridership + apartment trades | `src/ingest.py` or `src/mock_data.py` |
| 1 | Schema normalization (Korean → snake_case, dtypes) | `src/preprocess.py` |
| 2 | Entity resolution (station → coordinates → gu) | `src/preprocess.py` |
| 3 | Feature engineering (6 features per station) | `src/features.py` |
| 4 | Standardize + KMeans + UMAP | `src/cluster.py` |
| 5 | Monthly price index by gu | `src/preprocess.py` |
| 6 | Statistical tests (KS, ANOVA) | inline (scipy) |
| 7 | Chart export | `src/chart_functions.py` *(Phase 3)* |

**How to switch between mock and real data:** flip the `USE_MOCK` flag in the first cell. Everything downstream is identical because both sources produce the same schema.

In [ ]:
# --- Configuration ---------------------------------------------------------
USE_MOCK = True      # flip to False after your API keys are exported
SEED = 42
END_MONTH = "202605"
N_MONTHS = 24

import sys
from pathlib import Path

# Make the repo root importable regardless of where Jupyter was launched
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))

from src import cluster, features, mock_data, preprocess
from src.lawd_codes import FOCUS_LAWD_CODES, FOCUS_STATIONS

months = mock_data._month_range(END_MONTH, N_MONTHS)
print(f"Window: {months[0]} → {months[-1]}  ({len(months)} months)")

## Stage 0 — Ingestion

Two sources: Seoul Open Data (subway) and MOLIT (apartment trades). Both loaders are already in `src/ingest.py`; here we just call them (or the mock generator).

In [ ]:
if USE_MOCK:
    raw_subway = mock_data.generate_subway_hourly(months, seed=SEED)
    raw_trades = mock_data.generate_apartment_trades(
        list(FOCUS_LAWD_CODES.values()), months, seed=SEED
    )
else:
    from src import ingest
    raw_subway = ingest.fetch_subway_hourly(months, save_dir=REPO_ROOT / "data/raw")
    trades_frames = [
        ingest.fetch_apartment_trades(code, months, save_dir=REPO_ROOT / "data/raw")
        for code in FOCUS_LAWD_CODES.values()
    ]
    raw_trades = pd.concat(trades_frames, ignore_index=True)

print(f"raw subway (wide): {raw_subway.shape}")
print(f"raw trades:        {raw_trades.shape}")
raw_subway.head(2)

## Stage 1 — Schema normalization

The raw subway table is *wide* — 48 hour × direction columns with Korean names. We melt it to a *long tidy* table: one row per (station, month, hour, direction). This is the shape every downstream chart consumes.

In [ ]:
long = preprocess.normalize_subway_hourly(raw_subway)
long = preprocess.assign_station_id(long)
trades = preprocess.normalize_apartment_trades(raw_trades)

print(f"long ridership:  {long.shape}")
print(f"clean trades:    {trades.shape}\n")
long.head(3)

## Stage 3 — Feature engineering

Six features per station. See `src/features.py` for the exact definition of each; the notebook only shows the result.

In [ ]:
feats = features.compute_station_features(long)
print(f"Feature matrix: {feats.shape[0]} stations × {len(features.FEATURE_ORDER)} features")
feats.round(3)

## Stage 4 — Clustering

Sweep k = 2..8, pick the k with the highest silhouette score. Elbow (WCSS) is shown alongside for the diagnostic plot.

In [ ]:
result = cluster.choose_k_and_cluster(feats, k_range=(2, 8), random_state=SEED)
result = cluster.label_clusters_by_archetype(result)

print(f"k chosen: {result.k_chosen}")
print(f"Silhouette by k: { {k: round(v, 3) for k, v in result.silhouette_by_k.items()} }\n")
print("Cluster names:")
for idx, name in result.cluster_names.items():
    print(f"  cluster {idx}: {name}")

In [ ]:
# Focus-station attribution
for kr_name, meta in FOCUS_STATIONS.items():
    if kr_name in feats["station"].values:
        row_idx = feats.index[feats["station"] == kr_name][0]
        label = int(result.labels[row_idx])
        got = result.cluster_names[label]
        want = meta["cluster_hypothesis"]
        check = "✅" if want.lower() in got.lower() else "⚠️"
        print(f"  {check}  {kr_name:6}  →  {got:30}  (hypothesis: {want})")

## Stage 4b — 2D projection (UMAP)

The 6-feature space is embedded to 2D so we can plot every station as a dot colored by cluster. UMAP preserves local structure better than PCA for cluster visualization.

In [ ]:
try:
    result = cluster.add_umap_embedding(result, feats, random_state=SEED)
    print(f"UMAP embedding shape: {result.embedding_2d.shape}")
except ImportError as e:
    print(f"Skipping UMAP (not installed): {e}")

## Stage 5 — Monthly price index by gu

In [ ]:
price_idx = preprocess.build_monthly_price_index(trades)
print(f"Monthly price index: {price_idx.shape}")
price_idx.head(6)

## Persist processed tables for the interactive site

`src/build_interactive.py` will load these parquet files and export per-stage JSON snapshots that the browser reads.

In [ ]:
processed = REPO_ROOT / "data/processed"
processed.mkdir(parents=True, exist_ok=True)
long.to_parquet(processed / "ridership_long.parquet", index=False)
feats.to_parquet(processed / "features.parquet", index=False)
price_idx.to_parquet(processed / "price_index.parquet", index=False)

# Cluster labels + centroids saved as a small combined parquet
import pandas as pd
labels_df = pd.DataFrame({
    "station": feats["station"],
    "station_id": feats["station_id"],
    "cluster": result.labels,
    "cluster_name": [result.cluster_names[c] for c in result.labels],
})
labels_df.to_parquet(processed / "cluster_labels.parquet", index=False)
result.centroids_original.to_parquet(processed / "cluster_centroids.parquet")

print(f"Persisted to {processed}:")
for f in sorted(processed.glob("*.parquet")):
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")

---

**Next stage (Phase 3):** `chart_functions.py` — implements all 16 charts as reusable functions, consuming the parquet files above.

**Next stage (Phase 4):** `interactive/` — GitHub Actions builds precomputed JSON snapshots from these parquets and deploys to Pages.